# Week 1 Day 4 Challenge: more accuracy from fewer tokens

**The challenge:** find new ways to shorten the prompt that keep the accuracy, and a way to get high accuracy with as few tokens as possible.

**What I found:** the lab's prompt leaves out the text of the documents, and some claims can't be decided without it. Putting the documents back costs tokens, but the prompt carries more than that in things no decision uses, so you can have both. Prompt C below got DeepSeek V4.1 Flash right on every one of the 20 queue claims, decision and payout, in each of three runs, with 7% fewer input tokens than the lab's prompt and 16% less cost a run. On 20 held-out claims it was never tuned on, it got 98% right against 70% for the lab's prompt. It also did better than the lab's prompt on every other model I tried.

Decision and payout right:

| Model | Where | Lab prompt (A) | Prompt C | Input tokens a claim, A → C |
|---|---|---|---|---|
| DeepSeek V4.1 Flash | OpenRouter | 78% | **100%** | 1,169 → 1,083 |
| DeepSeek V4.1 Flash, held-out claims | OpenRouter | 70% | **98%** | 1,144 → 1,078 |
| Gemma 4 26B | OpenRouter | 57% | 65% | 1,237 → 1,131 |
| Nemotron 3 Nano 30B | OpenRouter | 50% | 62% | 1,211 → 1,145 |
| GPT-OSS 120B | OpenRouter | 43% | 62% | 1,210 → 1,173 |
| Gemma 4 E2B | This computer | 25% | 38% | 1,203 → 1,120 |
| Gemma 4 E2B, held-out claims | This computer | 28% | 43% | 1,211 → 1,089 |
| Gemma 4 E4B | This computer | 40% | 52% | 1,203 → 1,120 |
| Qwen 3.5 4B, one run | This computer | 35% | 45% | 1,207 → 1,124 |

For the small local models a different idea helps more: let the model judge and let code do the arithmetic (prompt D). On Gemma 4 E2B it took decision and payout from 25% to 45% on the queue and from 28% to 55% on the held-out claims. It isn't a rule for every small model, though: D's answers are longer, and Gemma 4 E4B ran out of room on more of them and did worse with D than with C.

The notebook runs four prompts on the lab's two best models (one on OpenRouter, one local), three runs each, then tries more models, then checks the winner on the held-out claims.

## Setup

The lab's data, the lab's helpers and an OpenRouter client. Results are kept in `results/`, one file for each model, so running the notebook again costs nothing for the runs that are already there.

In [1]:
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
from dotenv import load_dotenv
from openai import LengthFinishReasonError, OpenAI
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"), timeout=120, max_retries=5)

DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(held_out=False):
    """The claims and the handlers' decisions keyed by claim id, leaving out the manual claims as the lab does."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    labels = {decision["claim_id"]: decision for decision in decisions if decision["difficulty"] != "manual"}
    return [claim for claim in claims if claim["claim_id"] in labels], labels


claims, labels = get_sample_data()
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
DATASETS = {"queue": (claims, labels)}
print(f"{len(claims)} claims")

20 claims


## Part 1: Four prompts

Every variant uses the lab's way of scoring: the decision must match the handler's, and for "right" the payout must be within a dollar too.

- **A lab**: the lab's prompt, unchanged. The documents go in by name and type only.
- **B docs**: A with the text of the documents put back. Some claims can't be decided without it: in the queue, a hospital record shows that the customer saw a urologist about the same kidney stone five weeks before buying the policy, and two airline letters confirm a delay without giving its cause, which the Trip Delay section asks for.
- **C lean**: B, made smaller again. The Limits table keeps only the customer's plan and the rows for this benefit, the claim keeps only the fields a decision uses (no summary, no item categories, no document types), and the JSON has no spaces. It also puts back a few words the short wording dropped: the full wording says "a pair or a set, *such as a pair of shoes, a suit or a camera with its lens*, counts as one item", and the per-item limit can't be worked out right without them.
- **D split**: C without the Payment section or any money rows. The model judges and the code calculates. The model gives the decision, whether each claimed line is covered, how many separate items it is (three shirts are three items) and what it is, and what others have paid. `work_out_payout` does the Payment section's four steps. It doesn't put the lines of a set together, so a camera and its lens on two lines are two items.

In [3]:
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")


def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)

BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")


def lab_wording(claim):
    """The lab's wording: the limits, the claim's benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])


def policy_record(claim):
    policy = policies[claim["policy_id"]]
    return {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}


def lab_messages(claim):
    """Variant A: the lab's prompt, with the documents' names and types but not their text."""
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    user = f"# Policy record\n\n{json.dumps(policy_record(claim))}\n\n# Claim\n\n{json.dumps(details)}\n"
    return [{"role": "system", "content": f"{INSTRUCTIONS}\n{lab_wording(claim)}\n"}, {"role": "user", "content": user}]

In [4]:
def docs_messages(claim):
    """Variant B: the lab's prompt with the text of the documents put back."""
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    user = f"# Policy record\n\n{json.dumps(policy_record(claim))}\n\n# Claim\n\n{json.dumps(details)}\n"
    return [{"role": "system", "content": f"{INSTRUCTIONS}\n{lab_wording(claim)}\n"}, {"role": "user", "content": user}]

In [5]:
PLANS = ["economy", "premium", "first"]


def parse_limits(section):
    """The Limits table as {row: {plan: number}}."""
    rows = [line.strip("|").split("|") for line in section.splitlines() if line.startswith("| ") and "Economy" not in line]
    return {cells[0].strip(): dict(zip(PLANS, (float(cell.replace(",", "")) for cell in cells[1:]))) for cells in rows}


LIMITS = parse_limits(sections["Limits"])

LIMIT_ROWS = {
    "Cancellation and Interruption": ["Cancellation and Interruption maximum"],
    "Trip Delay": ["Trip Delay waiting period, hours", "Trip Delay maximum"],
    "Baggage Delay": ["Baggage Delay waiting period, hours", "Baggage Delay maximum"],
    "Baggage": ["Baggage maximum", "Per-item limit", "Deductible"],
    "Passport": ["Passport maximum"],
    "Emergency Medical": ["Emergency Medical maximum", "Deductible"],
}


def plan_limits(claim, rows):
    """The rows of the Limits table this claim needs, for the customer's plan only."""
    plan = policies[claim["policy_id"]]["plan"]
    lines = [f"- {row}: {LIMITS[row][plan]:,.0f}" for row in rows]
    if "Deductible" in rows:
        lines.append("The deductible comes off once per claim.")
    return f"## Limits on the {plan} plan\n\n" + "\n".join(lines)


# The short wording dropped the full wording's examples of a set, and a handler needs them for the per-item limit
SET_RULE = ("a pair or a set counts as one item", "a pair or a set, such as a pair of shoes, a suit or a camera with its lens, counts as one item")


def lean_wording(claim, payment=True):
    """The lab's wording with the Limits table cut to this plan and benefit. Without payment, only the waiting period is kept."""
    benefit = BENEFITS[claim["claim_type"]]
    rows = LIMIT_ROWS[benefit] if payment else [row for row in LIMIT_ROWS[benefit] if "waiting period" in row]
    parts = [opening.strip()] + ([plan_limits(claim, rows)] if rows else []) + [sections[benefit].strip().replace(*SET_RULE)]
    if policies[claim["policy_id"]]["add_ons"]:
        parts.append(sections["Add-ons"].strip())
    parts.append(sections["Never covered"].strip())
    if payment:
        parts.append(sections["Payment"].strip())
    parts.append(sections["Claims"].strip())
    return "\n\n".join(parts)


def lean_user_prompt(claim):
    """Only the fields a decision uses: the items without their category, and the documents with their text but not their type."""
    details = {key: claim[key] for key in ["claim_type", "submitted_on", "incident_date", "incident_location", "story", "amount_claimed"]}
    details["items"] = [{"description": item["description"], "amount": item["amount"]} for item in claim["items"]]
    details["documents"] = [{"name": document["name"], "content": document["content"]} for document in claim["documents"]]
    return f"# Policy record\n\n{json.dumps(policy_record(claim), separators=(',', ':'))}\n\n# Claim\n\n{json.dumps(details, separators=(',', ':'))}\n"


def lean_messages(claim):
    """Variant C: B with the limits cut to this plan and benefit, the fields a decision doesn't use left out, and the examples of a set put back."""
    return [{"role": "system", "content": f"{INSTRUCTIONS}\n{lean_wording(claim)}\n"}, {"role": "user", "content": lean_user_prompt(claim)}]

In [6]:
SPLIT_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Then go through the claimed lines in the order they are listed: say whether the wording covers each one, how many separate items it is, and what it is. Say what a carrier, hotel or other party has already paid. You don't work out the payout: it is calculated from your answers.
"""


class ItemCheck(BaseModel):
    """A handler's view of one claimed line."""

    description: str
    covered: bool = Field(description="True when the wording covers this line")
    pieces: int = Field(description="How many separate items the line is: three shirts are 3, a pair or a set is 1")
    kind: str = Field(description="What the item is, in one or two plain words, such as jacket or suitcase")


class ClaimCheck(BaseModel):
    """A claims handler's decision on one claim, with the facts the payout is worked out from."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    items: list[ItemCheck] = Field(description="One entry for each claimed line, in the order they are listed")
    paid_by_others: float = Field(description="What a carrier, hotel or other party has already paid towards the loss, or 0")


ELECTRONICS_PLUS_LIMIT = 1200  # the per-item limit for the listed devices on Electronics Plus, from the Add-ons section
DEVICE_WORDS = {"phone", "phones", "smartphone", "smartphones", "tablet", "tablets", "laptop", "laptops", "camera", "cameras", "lens", "lenses", "e-reader", "e-readers", "smartwatch", "smartwatches"}


def work_out_payout(claim, check):
    """The Payment section's four steps, in code, from the model's view of the claim. It doesn't put the lines of a set together."""
    if check.decision != "approve":
        return 0.0
    policy = policies[claim["policy_id"]]
    plan, benefit = policy["plan"], BENEFITS[claim["claim_type"]]
    total = 0.0
    for item, verdict in zip(claim["items"], check.items):
        if not verdict.covered:
            continue
        amount = item["amount"]
        if benefit == "Baggage":
            device = bool(DEVICE_WORDS & set(re.findall(r"[a-z-]+", verdict.kind.lower())))
            limit = ELECTRONICS_PLUS_LIMIT if device and "electronics_plus" in policy["add_ons"] else LIMITS["Per-item limit"][plan]
            amount = min(amount, limit * max(verdict.pieces, 1))
        total += amount
    total -= check.paid_by_others
    if "Deductible" in LIMIT_ROWS[benefit]:
        total -= LIMITS["Deductible"][plan]
    maximum = LIMITS[f"{benefit} maximum"][plan]
    if benefit == "Cancellation and Interruption":
        maximum = min(maximum, policy["trip"]["trip_cost"])
    return round(max(0.0, min(total, maximum)), 2)


def split_messages(claim):
    """Variant D: C without the Payment section or the money rows. The model judges, the code calculates."""
    return [{"role": "system", "content": f"{SPLIT_INSTRUCTIONS}\n{lean_wording(claim, payment=False)}\n"}, {"role": "user", "content": lean_user_prompt(claim)}]


VARIANTS = {
    "A lab": (lab_messages, ClaimDecision),
    "B docs": (docs_messages, ClaimDecision),
    "C lean": (lean_messages, ClaimDecision),
    "D split": (split_messages, ClaimCheck),
}

In [7]:
sizes = pd.DataFrame({name: [sum(len(message["content"]) for message in make_messages(claim)) for claim in claims] for name, (make_messages, _) in VARIANTS.items()})
print("Prompt length in characters, averaged over the 20 claims. Tokens are measured on the runs below.")
sizes.mean().round()

Prompt length in characters, averaged over the 20 claims. Tokens are measured on the runs below.


A lab      4246.0
B docs     5045.0
C lean     4297.0
D split    4200.0
dtype: float64

## Running the variants

`decide` asks one model about one claim. `run` does all 20 claims, in parallel on OpenRouter and one at a time on this computer, and repeats that for several runs. The same prompt can give a different answer each time, so a single run can be several claims out either way: in my lab runs, Gemma 4 E2B got 3 claims right one time and 8 the next with the same prompt. Each local run uses its run number as the seed.

Every call has a cap on its reply: `num_predict: 2048` on this computer and `max_tokens: 6000` on OpenRouter, where the reasoning counts too. Without one, a model can talk to itself and never finish: in my lab run Qwen 3.5 4B spent 17 minutes on one claim, and here Gemma 4 26B hit the cap on 13 of its 80 answers. An answer that hits the cap counts as "cut off" and wrong. OpenRouter calls also get a 90 second deadline, because a stalled request can stay open for many minutes.

In [8]:
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)


def with_deadline(call, seconds=90, attempts=3):
    """Run call in a background thread, giving up after seconds and trying again.

    The client's timeout is per read, and a stalled request on OpenRouter can stay open for many minutes."""
    for _ in range(attempts):
        box = {}

        def target():
            try:
                box["result"] = call()
            except Exception as error:
                box["error"] = error

        thread = threading.Thread(target=target, daemon=True)
        thread.start()
        thread.join(seconds)
        if "error" in box:
            raise box["error"]
        if "result" in box:
            return box["result"]
    raise TimeoutError(f"no answer in {attempts} tries of {seconds} seconds")


def decide(claim, variant, model, where, effort, seed):
    """Ask one model to decide one claim with one variant of the prompt."""
    make_messages, schema = VARIANTS[variant]
    start = time.time()
    if where == "OpenRouter":
        try:
            response = with_deadline(lambda: openrouter.chat.completions.parse(
                model=model,
                messages=make_messages(claim),
                response_format=schema,
                reasoning_effort=effort,
                max_tokens=6000,  # reasoning included; a model that keeps going would otherwise hit the deadline
                extra_body={"provider": {"require_parameters": True}},
            ))
            answer = response.choices[0].message.parsed
            input_tokens, cost = response.usage.prompt_tokens, response.usage.cost
        except LengthFinishReasonError as error:
            answer, input_tokens, cost = None, error.completion.usage.prompt_tokens, error.completion.usage.cost
    else:
        response = ollama.chat(
            model=model,
            messages=make_messages(claim),
            format=schema.model_json_schema(),
            think=False,
            options={"num_ctx": 8192, "num_predict": 2048, "seed": seed},
        )
        answer = None if response.done_reason == "length" else schema.model_validate_json(response.message.content)
        input_tokens, cost = response.prompt_eval_count, 0.0
    if answer is None:
        decision, payout, rationale = "cut off", 0.0, ""
    else:
        payout = work_out_payout(claim, answer) if schema is ClaimCheck else answer.payout
        decision, rationale = answer.decision, answer.rationale
    return {
        "model": model,
        "where": where,
        "claim_id": claim["claim_id"],
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
        "input_tokens": input_tokens,
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


def score(answer, label):
    """Whether the decision, and the decision with the payout to within a dollar, match the handler's."""
    decision_right = answer["decision"] == label["decision"]
    return {**answer, "difficulty": label["difficulty"], "decision_right": decision_right, "right": decision_right and abs(answer["payout"] - label["payout"]) <= 1}


def run(variant, model, where, effort=None, runs=1, data="queue"):
    """Run a model on every claim with one variant, runs times. Finished runs are kept in results/ and not paid for twice."""
    file = RESULTS / f"{model.replace('/', '_').replace(':', '_')}.json"
    cache = json.loads(file.read_text()) if file.exists() else {}
    claim_set, label_set = DATASETS[data]
    results, ran = [], False
    for seed in range(runs):
        key = f"{data}|{variant}|{effort}|{seed}"
        if key not in cache:
            ran = True
            with ThreadPoolExecutor(8 if where == "OpenRouter" else 1) as pool:
                answers = list(pool.map(lambda claim: decide(claim, variant, model, where, effort, seed), claim_set))
            scored = [score(answer, label_set[answer["claim_id"]]) for answer in answers]
            if data == "held_out":
                scored = [{key: value for key, value in result.items() if key != "rationale"} for result in scored]
            cache[key] = scored
            file.write_text(json.dumps(cache, indent=1))
        results += [{**result, "variant": variant, "effort": effort, "run": seed} for result in cache[key]]
    if where == "This computer" and ran:
        ollama.generate(model=model, keep_alive=0)  # unload the model, so the next one has the memory
    return results


def summarize(results):
    """One row per model and variant: accuracy averaged over the runs, the range across runs, tokens, cost and time."""
    frame = pd.DataFrame(results)
    frame["model"] = frame["model"].str.split("/").str[-1]
    per_run = frame.groupby(["model", "variant", "run"], sort=False).agg(
        decision=("decision_right", "mean"), right=("right", "mean"), tokens=("input_tokens", "mean"),
        cost=("cost", "sum"), seconds=("seconds", "mean"),
    )
    table = per_run.groupby(["model", "variant"], sort=False).agg(
        runs=("decision", "count"), decision=("decision", "mean"), right=("right", "mean"),
        lowest=("right", "min"), highest=("right", "max"), tokens=("tokens", "mean"), cost=("cost", "mean"), seconds=("seconds", "mean"),
    )
    return pd.DataFrame({
        "runs": table["runs"],
        "decision right (%)": (100 * table["decision"]).round(),
        "decision and payout right (%)": (100 * table["right"]).round(),
        "range across runs (%)": [f"{100 * low:.0f}-{100 * high:.0f}" for low, high in zip(table["lowest"], table["highest"])],
        "input tokens a claim": table["tokens"].round(),
        "cost a run ($)": table["cost"].round(4),
        "seconds a claim": table["seconds"].round(1),
    })


def by_claim(results, model):
    """How many runs got each claim right, decision and payout, for each variant."""
    frame = pd.DataFrame([result for result in results if result["model"] == model])
    return frame.pivot_table(index="claim_id", columns="variant", values="right", aggfunc=lambda right: f"{right.sum()}/{len(right)}", sort=False)

## Part 2: The four prompts on two models, three runs each

DeepSeek V4.1 Flash was the most accurate model in the lab and one of the cheapest. Gemma 4 E2B was the most accurate of the local models and the fastest.

In [9]:
DEEPSEEK = "deepseek/deepseek-v4.1-flash"
GEMMA = "gemma4:e2b-it-qat"

deepseek = [result for variant in VARIANTS for result in run(variant, DEEPSEEK, "OpenRouter", "low", runs=3)]

In [10]:
gemma = [result for variant in VARIANTS for result in run(variant, GEMMA, "This computer", runs=3)]

In [11]:
summarize(deepseek + gemma)

runs  decision right (%)  \
model               variant                             
deepseek-v4.1-flash A lab       3                85.0   
                    B docs      3               100.0   
                    C lean      3               100.0   
                    D split     3               100.0   
gemma4:e2b-it-qat   A lab       3                60.0   
                    B docs      3                67.0   
                    C lean      3                63.0   
                    D split     3                55.0   

                             decision and payout right (%)  \
model               variant                                  
deepseek-v4.1-flash A lab                             78.0   
                    B docs                            93.0   
                    C lean                           100.0   
                    D split                           92.0   
gemma4:e2b-it-qat   A lab                             25.0   
                    B docs                            25.0   
                    C lean                            38.0   
                    D split                           45.0   

                            range across runs (%)  input tokens a claim  \
model               variant                                               
deepseek-v4.1-flash A lab                   75-80                1169.0   
                    B docs                  90-95                1375.0   
                    C lean                100-100                1083.0   
                    D split                 90-95                1097.0   
gemma4:e2b-it-qat   A lab                   10-35                1203.0   
                    B docs                  20-30                1455.0   
                    C lean                  35-40                1120.0   
                    D split                 45-45                1081.0   

                             cost a run ($)  seconds a claim  
model               variant                                   
deepseek-v4.1-flash A lab            0.0201              7.7  
                    B docs           0.0201             10.3  
                    C lean           0.0169              6.5  
                    D split          0.0309             14.4  
gemma4:e2b-it-qat   A lab            0.0000              4.3  
                    B docs           0.0000              5.4  
                    C lean           0.0000              5.0  
                    D split          0.0000              5.0

### Claim by claim

How many of the three runs got each claim right, decision and payout.

In [12]:
by_claim(deepseek, DEEPSEEK)

variant,A lab,B docs,C lean,D split
claim_id,,,,
CLM-1001,3/3,3/3,3/3,3/3
CLM-1002,3/3,3/3,3/3,3/3
CLM-1003,3/3,3/3,3/3,3/3
CLM-1004,2/3,3/3,3/3,3/3
CLM-1005,3/3,3/3,3/3,3/3
CLM-1006,3/3,3/3,3/3,3/3
CLM-1007,3/3,3/3,3/3,3/3
CLM-1008,3/3,3/3,3/3,1/3
CLM-1009,3/3,3/3,3/3,3/3


In [13]:
by_claim(gemma, GEMMA)

variant,A lab,B docs,C lean,D split
claim_id,,,,
CLM-1001,3/3,3/3,3/3,3/3
CLM-1002,0/3,0/3,1/3,2/3
CLM-1003,2/3,0/3,2/3,3/3
CLM-1004,2/3,1/3,3/3,3/3
CLM-1005,3/3,2/3,3/3,3/3
CLM-1006,2/3,2/3,2/3,2/3
CLM-1007,0/3,1/3,1/3,2/3
CLM-1008,0/3,0/3,2/3,1/3
CLM-1009,0/3,1/3,3/3,0/3


## Part 3: More models

Three open-weight models on OpenRouter that the lab didn't try: Gemma 4 26B (a bigger relative of the local Gemmas), GPT-OSS 120B and the small Nemotron 3 Nano. They get the lab's prompt and C, two runs each. The two other local models from the lab get A, C and D, because D is meant for small models: Gemma 4 E4B twice, and Qwen 3.5 4B once, as it takes about 8 minutes a run on my laptop.

In [14]:
more_cloud = [
    result
    for model in ["google/gemma-4-26b-a4b-it", "nvidia/nemotron-3-nano-30b-a3b", "openai/gpt-oss-120b"]
    for variant in ["A lab", "C lean"]
    for result in run(variant, model, "OpenRouter", "low", runs=2)
]

In [15]:
more_local = [
    result
    for model, runs in [("gemma4:e4b", 2), ("qwen3.5:4b", 1)]
    for variant in ["A lab", "C lean", "D split"]
    for result in run(variant, model, "This computer", runs=runs)
]

In [16]:
summarize(more_cloud + more_local)

runs  decision right (%)  \
model                   variant                             
gemma-4-26b-a4b-it      A lab       2                72.0   
                        C lean      2                73.0   
nemotron-3-nano-30b-a3b A lab       2                70.0   
                        C lean      2                75.0   
gpt-oss-120b            A lab       2                60.0   
                        C lean      2                75.0   
gemma4:e4b              A lab       2                62.0   
                        C lean      2                68.0   
                        D split     2                45.0   
qwen3.5:4b              A lab       1                65.0   
                        C lean      1                65.0   
                        D split     1                70.0   

                                 decision and payout right (%)  \
model                   variant                                  
gemma-4-26b-a4b-it      A lab                             57.0   
                        C lean                            65.0   
nemotron-3-nano-30b-a3b A lab                             50.0   
                        C lean                            62.0   
gpt-oss-120b            A lab                             43.0   
                        C lean                            62.0   
gemma4:e4b              A lab                             40.0   
                        C lean                            52.0   
                        D split                           38.0   
qwen3.5:4b              A lab                             35.0   
                        C lean                            45.0   
                        D split                           40.0   

                                range across runs (%)  input tokens a claim  \
model                   variant                                               
gemma-4-26b-a4b-it      A lab                   55-60                1237.0   
                        C lean                  60-70                1131.0   
nemotron-3-nano-30b-a3b A lab                   50-50                1211.0   
                        C lean                  60-65                1145.0   
gpt-oss-120b            A lab                   40-45                1210.0   
                        C lean                  60-65                1173.0   
gemma4:e4b              A lab                   35-45                1203.0   
                        C lean                  50-55                1120.0   
                        D split                 35-40                1081.0   
qwen3.5:4b              A lab                   35-35                1207.0   
                        C lean                  45-45                1124.0   
                        D split                 40-40                1086.0   

                                 cost a run ($)  seconds a claim  
model                   variant                                   
gemma-4-26b-a4b-it      A lab            0.0142             17.2  
                        C lean           0.0171             18.6  
nemotron-3-nano-30b-a3b A lab            0.0063              6.0  
                        C lean           0.0070              6.4  
gpt-oss-120b            A lab            0.0024              2.1  
                        C lean           0.0025              2.6  
gemma4:e4b              A lab            0.0000             10.6  
                        C lean           0.0000             10.0  
                        D split          0.0000             11.4  
qwen3.5:4b              A lab            0.0000             12.2  
                        C lean           0.0000              7.3  
                        D split          0.0000             10.6

## Part 4: A check on claims the prompt was never tuned on

I built the variants looking at the 20 queue claims, so they could be fitted to those 20. The held-out set is 20 other claims (leaving out the manual ones, as the lab does). The notebook shows only the scores, and `results/` keeps each answer's decision and payout but not its rationale. DeepSeek gets A and C, Gemma 4 E2B gets A, C and D, two runs each.

In [17]:
DATASETS["held_out"] = get_sample_data(held_out=True)

held_out = [
    result
    for model, where, effort, variants in [(DEEPSEEK, "OpenRouter", "low", ["A lab", "C lean"]), (GEMMA, "This computer", None, ["A lab", "C lean", "D split"])]
    for variant in variants
    for result in run(variant, model, where, effort, runs=2, data="held_out")
]

In [18]:
summarize(held_out)

runs  decision right (%)  \
model               variant                             
deepseek-v4.1-flash A lab       2                70.0   
                    C lean      2                98.0   
gemma4:e2b-it-qat   A lab       2                65.0   
                    C lean      2                55.0   
                    D split     2                70.0   

                             decision and payout right (%)  \
model               variant                                  
deepseek-v4.1-flash A lab                             70.0   
                    C lean                            98.0   
gemma4:e2b-it-qat   A lab                             28.0   
                    C lean                            43.0   
                    D split                           55.0   

                            range across runs (%)  input tokens a claim  \
model               variant                                               
deepseek-v4.1-flash A lab                   65-75                1144.0   
                    C lean                 95-100                1078.0   
gemma4:e2b-it-qat   A lab                   20-35                1211.0   
                    C lean                  40-45                1089.0   
                    D split                 50-60                1050.0   

                             cost a run ($)  seconds a claim  
model               variant                                   
deepseek-v4.1-flash A lab            0.0211              9.1  
                    C lean           0.0192              7.5  
gemma4:e2b-it-qat   A lab            0.0000              3.2  
                    C lean           0.0000              4.0  
                    D split          0.0000              4.6

## What I found

**1. The lab's prompt cut one thing too many: the text of the documents.** Three queue claims can't be decided from the story and the document names. In one, the hospital's discharge summary shows the customer saw a urologist about the kidney stone five weeks before buying the policy, so it's a pre-existing condition and a decline. In the other two, the airline's delay certificate gives the times but not the cause, and the Trip Delay section needs the carrier to confirm the cause, so they are referrals. With the lab's prompt DeepSeek missed all three in every run. Putting the documents back (B) got every decision right, up from 85%, for 18% more input tokens.

**2. The short wording lost something too.** DeepSeek missed one more claim in every run with A and B: a lost suitcase with a camera body and a lens on separate lines, and a dinner jacket and dress trousers. The handler counted each pair as a set, under one per-item limit. The short wording only says "a pair or a set counts as one item". The full wording adds "such as a pair of shoes, a suit or a camera with its lens". With those few words back (C), DeepSeek got it right in every run. Cutting a document down can drop the examples that a rule depends on.

**3. A good part of the prompt is not used by any decision.** C pays for the documents and the example of a set by keeping only the customer's plan in the Limits table and only the rows for the claim's benefit, by leaving out the claim's summary (the story says it again), the items' categories and the documents' types, and by writing the JSON without spaces. C ends up 7% smaller than the lab's prompt and 21% smaller than B, and more accurate than both.

**4. For small models, let the code do the arithmetic.** Gemma 4 E2B often gets the decision right and the payout wrong. Across three runs it did that 21 times with A, 15 times with C and 6 times with D, where the model only says what is covered, how many items each line is and what each item is, and `work_out_payout` does the rest. D took Gemma 4 E2B from 25% to 45% on the queue, and from 28% to 55% on the held-out claims, the best of its prompts. It doesn't help every small model: the model goes through every claimed line, so the answers are longer, and Gemma 4 E4B hit the 2,048-token cap on 12 of its 40 D answers against 4 with C. It scored 38% with D and 52% with C. Qwen 3.5 4B scored 40% with D and 45% with C, in one run each. For a strong model D is no better than C (92% against 100% for DeepSeek): it adds new ways to fail, such as a guess at what the airline already paid, and it can't see a set split across two lines.

**5. Asking a small model to choose from a list can make it worse.** My first D asked for each item's kind from a fixed list (phone, tablet, laptop, camera, e-reader, smartwatch or other). Gemma 4 E2B and Gemma 4 26B both called a pair of headphones and a game console "e-reader". When the true answer isn't on the list, the structured output forces the closest word that is, and the next items copy it. A free answer ("headphones") that the code checks against the list worked better.

**6. One run is not enough.** The same prompt and model can be several claims apart from one run to the next: Gemma 4 E2B with A scored between 10% and 35% across three runs. Every number here is an average of two or three runs, and the range is in the tables.

**7. Cap every reply.** Without a cap a model can keep writing for ever: Qwen 3.5 4B spent 17 minutes on one claim in my lab run, and here Gemma 4 26B hit the 6,000-token cap on 13 of its 80 answers. A cap turns a hang into a wrong answer you can count. On OpenRouter a stalled request also needs a deadline of its own, because the client's timeout is per read.

### Caveats

- 20 queue claims and 20 held-out claims, two or three runs each. A difference of one claim is 5 points.
- I designed B, C and D looking at the queue claims and their answers. The held-out check is there to catch a prompt fitted to those 20, and C held up: 98% against 70% for the lab's prompt.
- Qwen 3.5 4B had one run per prompt, so its numbers are the least reliable.
- Local times depend on the computer. Mine is a 16 GB Apple M5 laptop.
- The runs kept in `results/` cost $0.44 on OpenRouter. With the runs I threw away while fixing D, the caps and the deadline, the whole challenge cost about $0.67.

In [19]:
spent = sum(result["cost"] for file in RESULTS.glob("*.json") for results in json.loads(file.read_text()).values() for result in results)
print(f"OpenRouter cost of every run kept in results/: ${spent:.2f}")

OpenRouter cost of every run kept in results/: $0.44
